In [3]:
import pandas as pd

Importing database and assigning it to variables.

data_dictionary.csv doesn't need important because it is just a descriptor of the column names

In [4]:
international_match_dataset = pd.read_csv("soccer_db/international_matches.csv")
current_world_cup_groups = pd.read_csv("soccer_db/2022_world_cup_groups.csv")
current_world_cup_matches = pd.read_csv("soccer_db/2022_world_cup_matches.csv")
world_cup_match_history = pd.read_csv("soccer_db/world_cup_matches.csv")
world_cups = pd.read_csv("soccer_db/world_cups.csv")

Preprocessing dataset to see if there are any null values

In [5]:
international_match_dataset.isnull().values.any()
current_world_cup_groups.isnull().values.any()
current_world_cup_matches.isnull().values.any()
world_cup_match_history.isnull().values.any()
world_cups.isnull().values.any()

np.True_

There are nulls so I want to see where and what they are to make sure they don't effect my queries

In [6]:
datasets = {
    "international_match_dataset": international_match_dataset,
    "current_world_cup_groups": current_world_cup_groups,
    "current_world_cup_matches": current_world_cup_matches,
    "world_cup_match_history": world_cup_match_history,
    "world_cups": world_cups
}

for name, df in datasets.items():
    nulls = df.isnull().sum()
    null_columns = nulls[nulls > 0]
    if not null_columns.empty:
        print(f"\n{name} has NULL values in:")
        print(null_columns)


international_match_dataset has NULL values in:
Winning Team       4170
Losing Team        4170
Win Conditions    17568
dtype: int64

current_world_cup_matches has NULL values in:
Host Team    16
dtype: int64

world_cup_match_history has NULL values in:
Winning Team      199
Losing Team       199
Win Conditions    838
dtype: int64

world_cups has NULL values in:
Winner          1
Runners-Up      1
Third           1
Fourth          1
Goals Scored    1
dtype: int64


All of these null values don't matter to dataset so its fine to have them

In [7]:
international_match_dataset.sample(5)

,ID,Tournament,Date,Home Team,Home Goals,Away Goals,Away Team,Winning Team,Losing Team,Win Conditions,Home Stadium
2101,2102,Friendly,1954/7/25,Curaçao,2,0,Netherlands,Curaçao,Netherlands,NaN,True
17284,17285,UEFA Euro,2021/7/6,Italy,1,1,Spain,NaN,NaN,Italy win on penalties,False
1652,1653,Friendly,1947/10/19,Catalonia,3,1,Spain,Catalonia,Spain,NaN,True
13282,13283,Friendly,2008/8/20,Morocco,3,1,Benin,Morocco,Benin,NaN,True
1483,1484,Friendly,1942/7/19,Bulgaria,0,3,Germany,Germany,Bulgaria,NaN,True


This data isn't needed to analyze the World Cup matches because it is international friendlies therefore I will ignore this set of data.

My 6-10 queries to find results from data:

1. 5 teams with most world cup match wins
2. 5 teams with most world cups
3. 5 most recent world cup top 4 countries
4. 5 countries with most goals in world cup
5. What countries have hosted the world cup the most
6. Make a table of every country to win a world cup how many

Query 1: Top 5 teams with most world cup match wins

I want to find this out because people always talk about which team has won the most world cups but not really about who has won the most matches. I would assume that these would correlate with teams that won the most world cups and hope to find that out with my second query

In [8]:
valid_matches = world_cup_match_history.dropna(subset=['Winning Team'])
win_counts = valid_matches['Winning Team'].value_counts()
top_5_winning_teams = win_counts.head(5)

print("Top 5 teams with the most World Cup match wins:")
print(top_5_winning_teams)

Top 5 teams with the most World Cup match wins:
Winning Team
Brazil       73
Germany      67
Italy        45
Argentina    43
France       34
Name: count, dtype: int64


Now I am going to compare this with most World Cup wins to see if my hypothesis is correct. Because of my soccer knowledge, I believe it is but need to be sure.

Query 2: 5 countries with the most world cups

In [9]:
tournament_wins = world_cups['Winner'].value_counts()
print("Countries with the most World Cup titles:")
print(tournament_wins.head())

Countries with the most World Cup titles:
Winner
Brazil        5
Italy         4
Germany FR    3
Uruguay       2
Argentina     2
Name: count, dtype: int64


This roughly adds up to my hypothesis. All of the teams with the most match wins also have a high amount of titles. Germany is listed as Germany FR because West Germany competed as "Germany FR". They won it 3 times, and Germany won it once as a whole country. This data is slightly misleading because it doesn't count Germany and Germany FR as the same entity. I will now combine them.

In [11]:
world_cups['Winner'] = world_cups['Winner'].replace('Germany FR', 'Germany')

In [12]:
tournament_wins = world_cups['Winner'].value_counts()
print("Countries with the most World Cup titles:")
print(tournament_wins.head())

Countries with the most World Cup titles:
Winner
Brazil       5
Italy        4
Germany      4
Uruguay      2
Argentina    2
Name: count, dtype: int64


This worked as intended showing the 4 total World Cups Germany has

Query 3: 5 most recent top 4 World Cup finishes

In [13]:
recent_top4 = world_cups.sort_values(by='Year', ascending=False)[['Year', 'Winner', 'Runners-Up', 'Third', 'Fourth']]
print("Top 4 teams from the 5 most recent World Cups:")
print(recent_top4.head(5))

Top 4 teams from the 5 most recent World Cups:
    Year   Winner   Runners-Up        Third    Fourth
21  2022      NaN          NaN          NaN       NaN
20  2018   France      Croatia      Belgium   England
19  2014  Germany    Argentina  Netherlands    Brazil
18  2010    Spain  Netherlands      Germany   Uruguay
17  2006    Italy       France      Germany  Portugal


The NaN values from the 2022 World Cup make sense because by the time this dataset was last updated, the 2022 World Cup was still ongoing.

I am now changing my code to list the most recent 5 not including the empty one.

In [14]:
recent_top4_minus_latest = world_cups.sort_values(by='Year', ascending=False)[['Year', 'Winner', 'Runners-Up', 'Third', 'Fourth']].iloc[1:6]
print("Top 4 teams from the 5 World Cups before the most recent:")
print(recent_top4_minus_latest)

Top 4 teams from the 5 World Cups before the most recent:
    Year   Winner   Runners-Up        Third          Fourth
20  2018   France      Croatia      Belgium         England
19  2014  Germany    Argentina  Netherlands          Brazil
18  2010    Spain  Netherlands      Germany         Uruguay
17  2006    Italy       France      Germany        Portugal
16  2002   Brazil      Germany       Turkey  Korea Republic


This works

Query 4: 5 countries with the most goals scored in the World Cup

First I have to combine the datavalues for Home Team and Away team to just Team and Home Goals and Away Goals to just Goals

In [21]:
host_counts = world_cups['Host Country'].value_counts()

most_common_host = host_counts.head(1)

print("Country that has hosted the most World Cups:")
print(most_common_host)

Country that has hosted the most World Cups:
Host Country
Mexico    2
Name: count, dtype: int64


This printed the country with the most. However, I know that Spain also has hosted 2 so I am changing query to also show all the countries with the highest amount.

In [22]:
host_counts = world_cups['Host Country'].value_counts()

max_host_count = host_counts.max()

top_hosts = host_counts[host_counts == max_host_count]

print("Countries that have hosted the most World Cups:")
print(top_hosts)

Countries that have hosted the most World Cups:
Host Country
Mexico     2
France     2
Brazil     2
Italy      2
Germany    2
Name: count, dtype: int64


This shows all countries that are tied for hosting it the most

Query 6: Make a table of all countries to win a world cup and how many theyve won

In [23]:
valid_wins = world_cups.dropna(subset=['Winner'])

win_counts = valid_wins['Winner'].value_counts().sort_values(ascending=False)

print("World Cup Wins by Country:")
print(win_counts)

World Cup Wins by Country:
Winner
Brazil       5
Italy        4
Germany      4
Uruguay      2
Argentina    2
France       2
England      1
Spain        1
Name: count, dtype: int64


This shows every country to win a world cup and how many theyve won.

In summary, certain teams like Brazil, Germany, Italy, and France who have all been very good teams are all top 5 on the results for most wins, matches played, and world cup. This shows that teams that have the best stats are always good and no team that has a few good years has a lot of good stats.